In [2]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,summarize,poly)
from sklearn.model_selection import train_test_split

In [19]:
from functools import partial
from sklearn.model_selection import \
     (cross_validate, KFold, ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

In [21]:
Auto = load_data('Auto')
Auto_train, Auto_valid = train_test_split(Auto, test_size = 196, random_state = 0)

In [29]:
hp1 = MS(['horsepower'])
X_train = hp1.fit_transform(Auto_train)
Y_train = Auto_train['mpg']
model = sm.OLS(Y_train, X_train)
results = model.fit()

In [31]:
X_valid = hp1.transform(Auto_valid)
Y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
np.mean((Y_valid - valid_pred)**2)

23.61661706966988

In [41]:
def evalMSE(terms, response, train, test):
    mm = MS(terms)
    X_train = mm. fit_transform(train)
    Y_train = train[response]
    X_test = mm.transform(test)
    Y_test = test[response]
    results = sm.OLS(Y_train,X_train).fit()
    test_pred = results.predict(X_test)
    return np.mean((Y_test - test_pred)**2)

In [43]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                     'mpg',
                      Auto_train,
                      Auto_valid)
MSE

array([23.61661707, 18.76303135, 18.79694163])

In [53]:
Auto_train, Auto_valid = train_test_split(Auto,test_size = 196, random_state = 3)
MSE = np.zeros(3)
for idx, degree in enumerate(range(1,4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],'mpg',Auto_train,Auto_valid)
MSE

array([20.75540796, 16.94510676, 16.97437833])

In [55]:
Auto_train, Auto_valid = train_test_split(Auto,test_size = 196, random_state = 10)
MSE = np.zeros(3)
for idx, degree in enumerate(range(1,4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],'mpg',Auto_train,Auto_valid)
MSE

array([23.06058834, 19.71779411, 19.70841616])

In [67]:
Auto_train, Auto_valid = train_test_split(Auto,test_size = 196, random_state = 42)
MSE = np.zeros(3)
for idx, degree in enumerate(range(1,4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],'mpg',Auto_train,Auto_valid)
MSE

array([25.57387819, 22.21802005, 22.66767544])

In [72]:
hp_model = sklearn_sm(sm.OLS, MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(hp_model,X,Y,cv=Auto.shape[0])
cv_err = np.mean(cv_results['test_score'])
cv_err

24.23151351792922

In [79]:
cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M,X,Y,cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.23151352, 19.24821312, 19.33498406, 19.4244303 , 19.03322411])

In [89]:
A = np.array([3, 5, 9])
B = np.array([2,4])
np.add.outer(A,B)

array([[ 5,  7],
       [ 7,  9],
       [11, 13]])

In [91]:
np.power.outer(A,B)

array([[   9,   81],
       [  25,  625],
       [  81, 6561]])

In [95]:
np.minimum.outer(A,B)

array([[2, 3],
       [2, 4],
       [2, 4]])

In [99]:
cv_error = np.zeros(5)
cv = KFold(n_splits=10,shuffle=True,random_state=0)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H,np.arange(d+1))
    M_CV = cross_validate(M,X,Y,cv=cv)
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.20766449, 19.18533142, 19.27626666, 19.47848402, 19.13719154])

In [105]:
ShuffleSplit?

Init signature:
ShuffleSplit(
    n_splits=10,
    *,
    test_size=None,
    train_size=None,
    random_state=None,
)
Docstring:     
Random permutation cross-validator.

Yields indices to split data into training and test sets.

Note: contrary to other cross-validation strategies, random splits
do not guarantee that all folds will be different, although this is
still very likely for sizeable datasets.

Read more in the :ref:`User Guide <ShuffleSplit>`.

For visualisation of cross-validation behaviour and
comparison between common scikit-learn split methods
refer to :ref:`sphx_glr_auto_examples_model_selection_plot_cv_indices.py`

Parameters
----------
n_splits : int, default=10
    Number of re-shuffling & splitting iterations.

test_size : float or int, default=None
    If float, should be between 0.0 and 1.0 and represent the proportion
    of the dataset to include in the test split. If int, represents the
    absolute number of test samples. If None, the value is set to the
    

In [111]:
validation = ShuffleSplit(n_splits = 1,
                         test_size = 196,
                         random_state = 0)
results = cross_validate(hp_model,Auto.drop(['mpg'],axis=1), Auto['mpg'], cv=validation)
results['test_score']

array([23.61661707])

In [115]:
validation = ShuffleSplit(n_splits = 10,
                         test_size = 196,
                         random_state = 0)
results = cross_validate(hp_model,Auto.drop(['mpg'],axis=1), Auto['mpg'], cv=validation)
results['test_score'].mean(), results['test_score'].std()

(23.802232661034164, 1.4218450941091831)

In [119]:
validation = ShuffleSplit(n_splits = 10,
                         test_size = 196,
                         random_state = 3)
results = cross_validate(hp_model,Auto.drop(['mpg'],axis=1), Auto['mpg'], cv=validation)
results['test_score'].mean(), results['test_score'].std()

(25.027740060359104, 3.1869365448544738)

In [123]:
validation = ShuffleSplit(n_splits = 100,
                         test_size = 196,
                         random_state = 3)
results = cross_validate(hp_model,Auto.drop(['mpg'],axis=1), Auto['mpg'], cv=validation)
results['test_score'].mean(), results['test_score'].std()

(24.503286198225233, 1.9152683536466621)

In [159]:
Portfolio = load_data('Portfolio')
def alpha_func(D,idx):
    cov_ = np.cov(D[['X','Y']].loc[idx], rowvar=False)
    return ((cov_[1,1] - cov_[0,1]) / (cov_[0,0]+cov_[1,1]-2*cov_[0,1]))

In [161]:
def alpha_func1(data, idx):
    X = data.loc[idx, 'X']
    Y = data.loc[idx, 'Y']
    return (np.var(Y, ddof=1) - np.cov(X, Y, ddof=1)[0, 1]) / \
           (np.var(X, ddof=1) + np.var(Y, ddof=1) - 2 * np.cov(X, Y, ddof=1)[0, 1])

In [163]:
alpha_func(Portfolio,range(100))

0.57583207459283

In [165]:
alpha_func1(Portfolio,range(100))

0.57583207459283

In [167]:
np.random.default_rng?

Docstring:
default_rng(seed=None)
Construct a new Generator with the default BitGenerator (PCG64).

    Parameters
    ----------
    seed : {None, int, array_like[ints], SeedSequence, BitGenerator, Generator}, optional
        A seed to initialize the `BitGenerator`. If None, then fresh,
        unpredictable entropy will be pulled from the OS. If an ``int`` or
        ``array_like[ints]`` is passed, then it will be passed to
        `SeedSequence` to derive the initial `BitGenerator` state. One may also
        pass in a `SeedSequence` instance.
        Additionally, when passed a `BitGenerator`, it will be wrapped by
        `Generator`. If passed a `Generator`, it will be returned unaltered.

    Returns
    -------
    Generator
        The initialized generator object.

    Notes
    -----
    If ``seed`` is not a `BitGenerator` or a `Generator`, a new `BitGenerator`
    is instantiated. This function does not manage a default global instance.

    See :ref:`seeding_and_entropy` 

In [169]:
np.random.Generator.choice?

Docstring:
choice(a, size=None, replace=True, p=None, axis=0, shuffle=True)

Generates a random sample from a given array

Parameters
----------
a : {array_like, int}
    If an ndarray, a random sample is generated from its elements.
    If an int, the random sample is generated from np.arange(a).
size : {int, tuple[int]}, optional
    Output shape.  If the given shape is, e.g., ``(m, n, k)``, then
    ``m * n * k`` samples are drawn from the 1-d `a`. If `a` has more
    than one dimension, the `size` shape will be inserted into the
    `axis` dimension, so the output ``ndim`` will be ``a.ndim - 1 +
    len(size)``. Default is None, in which case a single value is
    returned.
replace : bool, optional
    Whether the sample is with or without replacement. Default is True,
    meaning that a value of ``a`` can be selected multiple times.
p : 1-D array_like, optional
    The probabilities associated with each entry in a.
    If not given, the sample assumes a uniform distribution over a

In [171]:
rng = np.random.default_rng(0)
alpha_func(Portfolio,rng.choice(100,100,replace = True))

0.6074452469619004

In [197]:
def boost_SE(func,D,n=None, B=1000, seed = 0):
    rng = np.random.default_rng(seed)
    first_,second_ = 0 , 0
    n = n or D.shape[0]
    for _ in range(B) :
        idx = rng.choice(len(D),n,replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

In [199]:
alpha_SE = boost_SE(alpha_func,Portfolio,B=1000,seed=0)
alpha_SE

0.09118176521277699

In [219]:
def boost_OLS(model_matrix, response, D, idx):
    D_ = D.iloc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_, X_).fit().params

In [221]:
hp_func = partial(boost_OLS, MS(['horsepower']), 'mpg')

In [223]:
hp_func

functools.partial(<function boost_OLS at 0x000001D77C597C40>, ModelSpec(terms=['horsepower']), 'mpg')

In [225]:
rng = np.random.default_rng(0)
np.array([hp_func(Auto,rng.choice(392,392,replace = True))for _ in range(10)])

array([[39.88064456, -0.1567849 ],
       [38.73298691, -0.14699495],
       [38.31734657, -0.14442683],
       [39.91446826, -0.15782234],
       [39.43349349, -0.15072702],
       [40.36629857, -0.15912217],
       [39.62334517, -0.15449117],
       [39.0580588 , -0.14952908],
       [38.66688437, -0.14521037],
       [39.64280792, -0.15555698]])

In [227]:
hp_se = boost_SE(hp_func,
               Auto,
               B = 1000,
               seed = 10)
hp_se

intercept     0.848807
horsepower    0.007352
dtype: float64

In [230]:
hp_se = boost_SE(hp_func,
               Auto,
               B = 1000,
               seed = 0)
hp_se

intercept     0.857854
horsepower    0.007458
dtype: float64

In [232]:
hp_model.fit(Auto,Auto['mpg'])
model_se = summarize(hp_model.results_)['std err']
model_se

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

In [236]:
quad_model = MS([poly('horsepower',2,raw=True)])
quad_func = partial(boost_OLS,quad_model,'mpg')
boost_SE(quad_func,Auto,B=1000)

intercept                                  2.067840
poly(horsepower, degree=2, raw=True)[0]    0.033019
poly(horsepower, degree=2, raw=True)[1]    0.000120
dtype: float64

In [238]:
M = sm.OLS(Auto['mpg'], quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64